# ML Project Work Flow Case Study

## From churn problem to deployment-ready pipeline

This notebook demonstrates framing, validation, EDA, leakage-safe preprocessing, baselines, cross-validation, tuning, threshold selection, calibration, slice analysis, explainability, packaging, and monitoring.

**Decision:** rank active customers by 30-day churn risk so a capacity-limited retention team can prioritize outreach.

## 1. ML contract

- Unit: one active customer at weekly scoring time.
- Target: churn in the next 30 days.
- Features: information available at scoring time only.
- Primary metric: average precision for an imbalanced target.
- Operating metrics: precision and recall at the selected threshold.
- Guardrails: segment performance, capacity, complaints, and incremental retention.

In [ ]:
from pathlib import Path
import json, joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.calibration import calibration_curve
from sklearn.compose import ColumnTransformer
from sklearn.datasets import make_classification
from sklearn.dummy import DummyClassifier
from sklearn.impute import SimpleImputer
from sklearn.inspection import permutation_importance
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score,classification_report,confusion_matrix,precision_recall_curve,roc_auc_score
from sklearn.model_selection import GridSearchCV,train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder,StandardScaler

plt.style.use("seaborn-v0_8-whitegrid"); RANDOM_STATE=42

## 2. Create and validate data

In [ ]:
X,y=make_classification(n_samples=8000,n_features=8,n_informative=5,weights=[.78,.22],class_sep=1.1,random_state=RANDOM_STATE)
customers=pd.DataFrame(X,columns=[f"feature_{i}" for i in range(8)])
customers["plan"]=pd.qcut(customers.feature_0,3,labels=["Basic","Plus","Premium"])
customers["region"]=np.random.default_rng(42).choice(["West","Central","East"],len(customers))
customers.loc[customers.sample(frac=.04,random_state=1).index,"feature_2"]=np.nan
customers["churned"]=y
assert customers.churned.isin([0,1]).all() and len(customers)==8000
display(customers.head()); display(customers.isna().sum().sort_values(ascending=False).head())

## 3. Explore target and segments

In [ ]:
fig,axes=plt.subplots(1,2,figsize=(11,4))
customers.churned.value_counts(normalize=True).sort_index().plot.bar(ax=axes[0],color=["#64748b","#0369a1"])
customers.groupby("plan",observed=True).churned.mean().plot.bar(ax=axes[1],color="#059669")
axes[0].set(title="Target balance",ylabel="Share"); axes[1].set(title="Observed churn by plan",ylabel="Rate")
plt.tight_layout(); plt.show()

## 4. Split before preprocessing

The test set is isolated before learning imputations, encodings, scales, coefficients, or thresholds. Time-dependent production data should generally use a forward split.

In [ ]:
target=customers.pop("churned")
X_train,X_test,y_train,y_test=train_test_split(customers,target,test_size=.2,stratify=target,random_state=RANDOM_STATE)
X_fit,X_valid,y_fit,y_valid=train_test_split(X_train,y_train,test_size=.25,stratify=y_train,random_state=RANDOM_STATE)
numeric=[x for x in customers if x.startswith("feature")]; categorical=["plan","region"]
preprocess=ColumnTransformer([("num",Pipeline([("impute",SimpleImputer(strategy="median")),("scale",StandardScaler())]),numeric),("cat",OneHotEncoder(handle_unknown="ignore"),categorical)])
pipeline=Pipeline([("preprocess",preprocess),("model",LogisticRegression(max_iter=2000,class_weight="balanced"))])

## 5. Baseline and cross-validated tuning

In [ ]:
baseline=DummyClassifier(strategy="prior").fit(X_fit,y_fit)
baseline_prob=baseline.predict_proba(X_valid)[:,1]
print(f"Baseline average precision: {average_precision_score(y_valid,baseline_prob):.3f}")
search=GridSearchCV(pipeline,{"model__C":[.03,.1,.3,1,3,10]},scoring="average_precision",cv=5,n_jobs=-1,return_train_score=True)
search.fit(X_fit,y_fit)
results=pd.DataFrame(search.cv_results_).sort_values("rank_test_score")
display(results[["param_model__C","mean_train_score","mean_test_score","std_test_score"]].head())

## 6. Select the operating threshold

In [ ]:
valid_prob=search.predict_proba(X_valid)[:,1]
precision,recall,thresholds=precision_recall_curve(y_valid,valid_prob)
f2=5*precision*recall/(4*precision+recall+1e-12); index=np.nanargmax(f2[:-1]); threshold=float(thresholds[index])
print(f"Validation ROC AUC: {roc_auc_score(y_valid,valid_prob):.3f}")
print(f"Validation average precision: {average_precision_score(y_valid,valid_prob):.3f}")
print(f"Selected threshold: {threshold:.3f}")
fig,ax=plt.subplots(figsize=(8,4)); ax.plot(thresholds,precision[:-1],label="Precision"); ax.plot(thresholds,recall[:-1],label="Recall"); ax.axvline(threshold,color="#dc2626",ls="--"); ax.legend(); ax.set(xlabel="Threshold",title="Business trade-off"); plt.show()

## 7. Final held-out evaluation

In [ ]:
model=search.best_estimator_.fit(X_train,y_train)
prob=model.predict_proba(X_test)[:,1]; pred=(prob>=threshold).astype(int)
print(f"Test ROC AUC: {roc_auc_score(y_test,prob):.3f}")
print(f"Test average precision: {average_precision_score(y_test,prob):.3f}")
print(classification_report(y_test,pred))
display(pd.DataFrame(confusion_matrix(y_test,pred),index=["Actual stay","Actual churn"],columns=["Predict stay","Predict churn"]))

## 8. Calibration, slices, and explainability

In [ ]:
fraction,mean_score=calibration_curve(y_test,prob,n_bins=10,strategy="quantile")
fig,axes=plt.subplots(1,2,figsize=(11,4)); axes[0].plot([0,1],[0,1],ls="--"); axes[0].plot(mean_score,fraction,marker="o"); axes[0].set(title="Calibration",xlabel="Predicted",ylabel="Observed")
importance=permutation_importance(model,X_test,y_test,scoring="average_precision",n_repeats=6,random_state=42,n_jobs=-1)
imp=pd.DataFrame({"feature":X_test.columns,"importance":importance.importances_mean}).sort_values("importance")
axes[1].barh(imp.feature,imp.importance,color="#059669"); axes[1].set(title="Permutation importance"); plt.tight_layout(); plt.show()
scored=X_test[["plan","region"]].copy(); scored["actual"]=y_test.to_numpy(); scored["score"]=prob; scored["prediction"]=pred
display(scored.groupby("plan",observed=True).agg(customers=("actual","size"),actual_rate=("actual","mean"),mean_score=("score","mean"),positive_rate=("prediction","mean")))

Importance is predictive, not causal. Slice summaries are diagnostics, not a complete fairness assessment.

## 9. Package the complete inference pipeline

In [ ]:
OUT=Path("outputs/notebook_case_study"); OUT.mkdir(parents=True,exist_ok=True)
joblib.dump(model,OUT/"churn_pipeline.joblib")
metadata={"target":"30_day_churn","threshold":threshold,"roc_auc":roc_auc_score(y_test,prob),"average_precision":average_precision_score(y_test,prob),"features":list(X_test.columns)}
(OUT/"model_metadata.json").write_text(json.dumps(metadata,indent=2)); imp.to_csv(OUT/"feature_importance.csv",index=False)
print(metadata)

## 10. Deployment, monitoring, and retraining

**Deployment:** weekly batch scoring, schema validation, versioned code/data/features/model/threshold, shadow comparison, rollback artifact.

**Monitoring:** system latency and failures; schema and missingness; feature and score drift; delayed ranking, calibration, and slice metrics; incremental retention measured with randomized holdouts.

**Retraining:** after sufficient new labels or validated degradation. Compare challenger and champion on a frozen temporal backtest.

### Limitations

- Synthetic data lacks temporal and operational complexity.
- The F2 threshold is illustrative; production should encode real costs and capacity.
- Logistic regression should be compared with a nonlinear challenger.
- Churn risk is not the same as persuadability; outreach needs causal testing.

### Conclusion

A successful ML project is a controlled decision system—not merely a fitted estimator. Framing, data, validation, deployment, monitoring, and feedback must form one reproducible loop.